# 🎙️ Thai GigaSpeech2 Dataset Processing Pipeline

## 📋 Overview

This notebook provides a comprehensive pipeline for processing the **Thai GigaSpeech2** dataset with:
- 🎯 **Transcript Integration** - Automatic loading and matching of Thai transcripts
- 🔄 **Streaming Processing** - Efficient handling of large datasets with streaming capabilities
- 📊 **Audio Analysis** - Visualization and analysis tools for audio data

## 🚀 Key Features

1. **Automated Transcript Loading** from Hugging Face Hub
2. **Enhanced Data Schema** combining audio and transcripts
3. **Visual Audio Analysis** with waveforms and spectrograms

---

### 🔧 Environment Setup

First, let's configure the environment to suppress unnecessary warnings and set up logging preferences:

In [1]:
# Disable warnings and configure logging
import warnings
import os
import logging

# Suppress all warnings
warnings.filterwarnings('ignore')

# Disable Lightning warnings specifically
os.environ['PYTORCH_LIGHTNING_SUPPRESS_WARNINGS'] = '1'
os.environ['HYDRA_FULL_ERROR'] = '0'

# Disable specific PyTorch Lightning logging
logging.getLogger("pytorch_lightning").setLevel(logging.ERROR)
logging.getLogger("lightning").setLevel(logging.ERROR)
logging.getLogger("lightning.pytorch").setLevel(logging.ERROR)

# Also disable transformers warnings if using Hugging Face
logging.getLogger("transformers").setLevel(logging.ERROR)

# Optional: Disable all INFO level logging
logging.basicConfig(level=logging.WARNING)

print("✅ Warnings and verbose logging disabled")

✅ Warnings and verbose logging disabled


In [2]:
# Disable warnings and configure logging - MUST RUN FIRST
import warnings
import os
import logging
import sys

# Suppress all warnings
warnings.filterwarnings('ignore')

# Disable Lightning warnings via environment variables (must be set before importing)
os.environ['PYTORCH_ENABLE_MPS_FALLBACK'] = '1'
os.environ['PYTORCH_LIGHTNING_SUPPRESS_WARNINGS'] = '1'
os.environ['HYDRA_FULL_ERROR'] = '0'
os.environ['TOKENIZERS_PARALLELISM'] = 'false'

# Create a null handler to suppress all logging
class NullHandler(logging.Handler):
    def emit(self, record):
        pass

# Configure root logger to suppress everything below WARNING
logging.getLogger().setLevel(logging.WARNING)
logging.getLogger().addHandler(NullHandler())

# Specifically target the problematic loggers
problematic_loggers = [
    "pytorch_lightning",
    "pytorch_lightning.utilities.migration.utils",
    "pytorch_lightning.utilities",
    "lightning",
    "lightning.pytorch",
    "lightning_fabric",
    "transformers",
    "torch",
    "torch.nn.modules.module",
]

for logger_name in problematic_loggers:
    logger = logging.getLogger(logger_name)
    logger.setLevel(logging.ERROR)
    logger.propagate = False
    logger.handlers = [NullHandler()]

# Redirect stdout temporarily during imports
class SuppressStdout:
    def __enter__(self):
        self._original_stdout = sys.stdout
        sys.stdout = open(os.devnull, 'w')
        return self

    def __exit__(self, exc_type, exc_val, exc_tb):
        sys.stdout.close()
        sys.stdout = self._original_stdout

print("✅ Aggressive warning suppression enabled - Lightning migration messages will be hidden")

✅ Aggressive warning suppression enabled - Lightning migration messages will be hidden


## 1️⃣ Dataset Loading

### 📥 Loading Thai GigaSpeech2 Dataset

We begin by loading the Thai subset of the GigaSpeech2 dataset in streaming mode. This allows us to process large amounts of data without loading everything into memory at once.

**Key Parameters:**
- `data_files`: Specifies the Thai training data location
- `streaming=True`: Enables memory-efficient streaming processing
- Returns an `IterableDataset` with audio waveforms and metadata

In [3]:
from datasets import load_dataset

dataset = load_dataset(
    "speechcolab/gigaspeech2",
    data_files={'train': 'data/th/train/*.tar.gz'},
    split='train',
    streaming=True
)

Resolving data files:   0%|          | 0/193 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/193 [00:00<?, ?it/s]

### 📊 Dataset Structure

The loaded dataset contains:
- **wav**: Audio waveform data with array and sampling rate
- **__key__**: Unique identifier for each audio segment
- **__url__**: Source URL of the audio file

In [4]:
print(dataset)

IterableDataset({
    features: ['wav', '__key__', '__url__'],
    num_shards: 193
})


## 2️⃣ Transcript Loading

### 📝 Loading Thai Transcripts from TSV File

The transcripts are stored separately in a TSV (Tab-Separated Values) file. This function downloads and parses the transcript file from Hugging Face Hub, creating a mapping from segment IDs to their corresponding Thai text transcriptions.

**Process:**
1. Downloads `train_refined.tsv` from the dataset repository
2. Parses the TSV file to extract segment ID and transcript pairs
3. Returns a dictionary for fast transcript lookup

In [5]:
from huggingface_hub import hf_hub_download
import csv
import json
from pprint import pprint

def load_thai_transcripts():
    """Load Thai transcripts from TSV file"""
    try:
        # Download the Thai transcript file
        tsv_path = hf_hub_download(
            repo_id="speechcolab/gigaspeech2",
            filename="data/th/train_refined.tsv",
            repo_type="dataset"
        )
        
        # Parse TSV file
        transcripts = {}
        with open(tsv_path, 'r', encoding='utf-8') as f:
            reader = csv.reader(f, delimiter='\t')
            for row in reader:
                if len(row) >= 2:
                    segment_id = row[0].strip()
                    transcript = row[1].strip()
                    transcripts[segment_id] = transcript
        
        print(f"Loaded {len(transcripts)} Thai transcripts")
        return transcripts
    
    except Exception as e:
        print(f"Failed to load transcripts: {e}")
        return {}


In [6]:
transcripts = load_thai_transcripts()

Loaded 9904271 Thai transcripts


## 4️⃣ Data Processing Pipeline

### 🔄 Dataset Iteration and Enhancement

This section demonstrates how to iterate through the dataset and enhance each sample with:
- Extracted segment IDs from the file paths
- Matched transcripts from our loaded dictionary
- Rich metadata for downstream processing

The example processes the first 50 samples to showcase the data structure.

def stream_thai_gigaspeech2_with_transcripts(dataset, transcripts, max_samples=10):
    """
    Stream dataset with transcripts added to schema
    This creates a new data schema that includes transcripts
    
    Args:
        dataset: The input dataset
        transcripts: Dictionary of transcripts
        max_samples: Maximum number of samples to process
    """
    
    dataset_iter = iter(dataset)
    count = 0
    
    for sample in dataset_iter:
        # Extract segment ID for transcript lookup
        segment_id = None
        key = sample.get('__key__', '')
        if key:
            # Format: "0/685/0-685-9" -> extract "0-685-9"  
            parts = key.split('/')
            if parts:
                segment_id = parts[-1]
        
        # Get transcript
        transcript = ""
        if segment_id and segment_id in transcripts:
            transcript = transcripts[segment_id]
        
        # Create new schema that includes transcript
        enhanced_sample = {
            'segment_id': segment_id,
            'audio': sample.get('wav', {}),
            'transcript': transcript,
            'metadata': {
                'original_key': sample.get('__key__', ''),
                'url': sample.get('__url__', ''),
                'has_transcript': bool(transcript)
            }
        }
        
        yield enhanced_sample
        
        count += 1
        if count >= max_samples:
            break

# Test the enhanced streaming function
print("Testing enhanced streaming with transcript integration:")
print("=" * 70)

for i, sample in enumerate(stream_thai_gigaspeech2_with_transcripts(dataset, transcripts, max_samples=10)):
    print(f"Sample {i+1}:")
    print(f"  Segment ID: {sample['segment_id']}")
    print(f"  Has transcript: {sample['metadata']['has_transcript']}")
    print(f"  Transcript preview: {sample['transcript'][:50]}{'...' if len(sample['transcript']) > 50 else ''}")

print("\n" + "=" * 70)
print("✓ Successfully integrated transcripts with audio data")

In [ ]:
def stream_thai_gigaspeech2_with_transcripts(dataset, transcripts, max_samples=10):
    """
    Stream dataset with transcripts added to schema
    This creates a new data schema that includes transcripts
    Only yields samples that have transcripts
    
    Args:
        dataset: The input dataset
        transcripts: Dictionary of transcripts
        max_samples: Maximum number of samples to process
    """
    
    dataset_iter = iter(dataset)
    count = 0
    
    for sample in dataset_iter:
        # Extract segment ID for transcript lookup
        segment_id = None
        key = sample.get('__key__', '')
        if key:
            # Format: "0/685/0-685-9" -> extract "0-685-9"  
            parts = key.split('/')
            if parts:
                segment_id = parts[-1]
        
        # Get transcript
        transcript = ""
        if segment_id and segment_id in transcripts:
            transcript = transcripts[segment_id]
        
        # Skip samples without transcripts
        if not transcript:
            continue
        
        # Create new schema that includes transcript
        enhanced_sample = {
            'segment_id': segment_id,
            'audio': sample.get('wav', {}),
            'transcript': transcript,
            'metadata': {
                'original_key': sample.get('__key__', ''),
                'url': sample.get('__url__', ''),
                'has_transcript': bool(transcript)
            }
        }
        
        yield enhanced_sample
        
        count += 1
        if count >= max_samples:
            break

# Test the enhanced streaming function
print("Testing enhanced streaming with transcript integration:")
print("=" * 70)

for i, sample in enumerate(stream_thai_gigaspeech2_with_transcripts(dataset, transcripts, max_samples=10)):
    print(f"Sample {i+1}:")
    print(f"  Segment ID: {sample['segment_id']}")
    print(f"  Has transcript: {sample['metadata']['has_transcript']}")
    print(f"  Transcript preview: {sample['transcript'][:50]}{'...' if len(sample['transcript']) > 50 else ''}")

print("\n" + "=" * 70)
print("✓ Successfully integrated transcripts with audio data")

### 🎯 Simplified Streaming (Transcripts Only)

For cases where speaker identification is not needed, this lightweight function provides:
- Transcript matching only
- Reduced computational overhead
- Same enhanced data schema structure

### 🎯 Basic Data Iteration Example

This example shows how to iterate through the dataset and access the basic features:

In [12]:
def stream_thai_gigaspeech2_with_transcripts(dataset, transcripts, max_samples=10):
    """
    Stream dataset with transcripts added to schema
    This creates a new data schema that includes transcripts
    Only yields samples that have transcripts
    """
    
    dataset_iter = iter(dataset)
    count = 0
    
    for sample in dataset_iter:
        # Extract segment ID for transcript lookup
        segment_id = None
        key = sample.get('__key__', '')
        if key:
            # Format: "0/685/0-685-9" -> extract "0-685-9"  
            parts = key.split('/')
            if parts:
                segment_id = parts[-1]
        
        # Get transcript
        transcript = ""
        if segment_id and segment_id in transcripts:
            transcript = transcripts[segment_id]
        
        # Skip samples without transcripts
        if not transcript:
            continue
        
        # Create new schema that includes transcript
        enhanced_sample = {
            'segment_id': segment_id,
            'audio': sample.get('wav', {}),
            'transcript': transcript,
            'metadata': {
                'original_key': sample.get('__key__', ''),
                'url': sample.get('__url__', ''),
                'has_transcript': bool(transcript)
            }
        }
        
        yield enhanced_sample
        
        count += 1
        if count >= max_samples:
            break

# Test the streaming function
print("Testing enhanced streaming with transcripts:")
print("=" * 60)

for i, sample in enumerate(stream_thai_gigaspeech2_with_transcripts(dataset, transcripts, max_samples=10)):
    print(f"\nSample {i+1}:")
    print(f"  Segment ID: {sample['segment_id']}")
    print(f"  Has transcript: {sample['metadata']['has_transcript']}")
    print(f"  Transcript length: {len(sample['transcript'])}")
    print(f"  Transcript preview: {sample['transcript'][:80]}{'...' if len(sample['transcript']) > 80 else ''}")
    print(f"  Audio shape: {sample['audio']['array'].shape if 'array' in sample['audio'] else 'N/A'}")
    print(f"  Sample rate: {sample['audio']['sampling_rate'] if 'sampling_rate' in sample['audio'] else 'N/A'}")

Testing enhanced streaming with transcripts:

Sample 1:
  Segment ID: 0-685-25
  Has transcript: True
  Transcript length: 49
  Transcript preview: พังยับแบบนี้หลังคายู่ห้องโดยสารยู่เข้าไปเพราะอะไร
  Audio shape: (63683,)
  Sample rate: 16000

Sample 2:
  Segment ID: 0-685-54
  Has transcript: True
  Transcript length: 39
  Transcript preview: อย่างที่บอกว่าต้องสอบปากคําคนขับแท็กซี่
  Audio shape: (33602,)
  Sample rate: 16000

Sample 3:
  Segment ID: 0-685-58
  Has transcript: True
  Transcript length: 54
  Transcript preview: แล้วก็ดําเนินการตามขั้นตอนของกฎหมายอีกครั้งหนึ่งนะครับ
  Audio shape: (51522,)
  Sample rate: 16000

Sample 4:
  Segment ID: 0-685-3
  Has transcript: True
  Transcript length: 26
  Transcript preview: เสียหลักไปชนท้ายกับรถ๑๐ล้อ
  Audio shape: (36482,)
  Sample rate: 16000

Sample 5:
  Segment ID: 0-685-35
  Has transcript: True
  Transcript length: 48
  Transcript preview: หลักฐานนะครับทีนี้เนี่ยยังมีผู้บาดเจ็บสองรายนะฮะ
  Audio shape: (59523,)
  Sample rate: 1

## 6️⃣ Testing & Verification

### ✅ Testing Transcript Integration

This section verifies the complete pipeline by:
1. Processing multiple samples with transcript matching
2. Displaying sample details for verification
3. Confirming all components are functioning correctly